### Imports and Loading Model

In [1]:
# Import
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import chromadb

C:\Users\Raghu Sanivarapu\venvs\simple-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2962.08it/s]

### Knowledge Base

In [3]:
# 17 IT help desk articles, each one starting with a "## " heading. Some share a word with a different meaning
# (key, token, freeze, monitor, license), and some hold near-identical facts (two Outlook error codes,
# two room display codes, and a retired help desk number next to the current one)
knowledge_base = """
## BitLocker Recovery Key

If your laptop asks for a BitLocker recovery key at startup, call the help desk and read out the key ID shown on the screen.

## Sticky Keyboard Keys

If keyboard keys stick or repeat, turn the laptop off and clean under the keys with compressed air.

## Office Keys

Keys for desk drawers and lockers are kept at reception. Sign the key log when you borrow one.

## Lost Security Token

If you lose your YubiKey security token, report it to the security team the same day so it can be disabled.

## GitHub Token Expired

GitHub personal access tokens expire after 90 days. Create a new one under Settings, then Developer settings.

## Laptop Freezes

If your laptop freezes, hold the power button for 10 seconds to restart it. If it happens every day, open a ticket.

## Change Freeze

No production changes are allowed during the change freeze, from December 15 to January 5.
Emergency fixes need approval from the CTO.

## Second Monitor

If a second monitor is not detected, check the cable, then press Windows+P and choose Extend.

## Server Monitoring

Server monitoring alerts page the on-call engineer. Acknowledge an alert within 15 minutes.

## Help Desk Phone (Retired)

The old help desk number, 555-0100, was retired in 2025 and is no longer answered.

## Help Desk Contact

Call the help desk at 555-0199, or open a ticket in the IT portal. Phones are answered from 7 AM to 7 PM.

## Outlook Error 0x800CCC0E

Outlook error 0x800CCC0E means Outlook cannot reach the mail server. Check your network and restart Outlook.

## Outlook Error 0x800CCC0F

Outlook error 0x800CCC0F means the connection dropped while sending. Wait a minute, then send the email again.

## Room Everest Display

To share your screen on the Room Everest display, enter the code 4417 on the wall panel.

## Room Denali Display

To share your screen on the Room Denali display, enter the code 4471 on the wall panel.

## Software License

If Adobe says your license expired, sign out of Creative Cloud and sign back in. Licenses renew every March.

## Parking Permit

To get a parking permit, upload a copy of your driving license to the facilities portal.
"""

# One article per heading: the heading becomes its "title", the rest its text
articles = MarkdownHeaderTextSplitter([("##", "title")]).split_text(knowledge_base)
print("Articles:", len(articles))

Articles: 17


### Chunking

In [4]:
# Cut each article into chunks of up to 150 characters: a longer article splits at its line breaks, a shorter one stays whole
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)
chunks = splitter.split_documents(articles)   # each chunk keeps its article's title

print("Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"C{i+1} [{chunk.metadata['title']}] {chunk.page_content}")

Chunks: 17
C1 [BitLocker Recovery Key] If your laptop asks for a BitLocker recovery key at startup, call the help desk and read out the key ID shown on the screen.
C2 [Sticky Keyboard Keys] If keyboard keys stick or repeat, turn the laptop off and clean under the keys with compressed air.
C3 [Office Keys] Keys for desk drawers and lockers are kept at reception. Sign the key log when you borrow one.
C4 [Lost Security Token] If you lose your YubiKey security token, report it to the security team the same day so it can be disabled.
C5 [GitHub Token Expired] GitHub personal access tokens expire after 90 days. Create a new one under Settings, then Developer settings.
C6 [Laptop Freezes] If your laptop freezes, hold the power button for 10 seconds to restart it. If it happens every day, open a ticket.
C7 [Change Freeze] No production changes are allowed during the change freeze, from December 15 to January 5.
Emergency fixes need approval from the CTO.
C8 [Second Monitor] If a second monitor

### Indexing Data

In [5]:
texts = [chunk.page_content for chunk in chunks]
titles = [chunk.metadata for chunk in chunks]
ids = [f"C{i+1}" for i in range(len(chunks))]

# Setup Chroma client and collection, in memory
chroma_client = chromadb.Client()
# Drop existing collection if it already exists
try:
    chroma_client.delete_collection("it_kb")
except Exception:
    pass

# Now create fresh: "cosine" makes the distance 1 - cosine similarity
collection = chroma_client.create_collection("it_kb", configuration={"hnsw": {"space": "cosine"}})

# Add chunks with embeddings
embeddings = model.encode(texts).tolist()
collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=titles)

print("Indexed chunks:", collection.count())

Indexed chunks: 17


### Basic RAG Query

In [6]:
# Each question, then the article it should come from. Basic search ranks by closeness of meaning only,
# so a chunk that shares words with the question ("laptop", "key") can beat the right one
questions = {
    "My keyboard key is stuck": "Sticky Keyboard Keys",
    "My new laptop asks for a key": "BitLocker Recovery Key",
    "Can I deploy code during the freeze?": "Change Freeze",
    "What is the help desk phone number?": "Help Desk Contact",
    "What does Outlook error 0x800CCC0F mean?": "Outlook Error 0x800CCC0F",
}

for query, expected in questions.items():
    query_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=3)

    top_title = results["metadatas"][0][0]["title"]
    verdict = "RIGHT" if top_title == expected else "WRONG"
    print()
    print(f"Question: {query}")
    print(f"Should come from: {expected}  ->  top result is {verdict}")
    for chunk_id, text, meta, distance in zip(results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"  {chunk_id} [{meta['title']}] similarity={1 - distance:.4f}: {text}")


Question: My keyboard key is stuck
Should come from: Sticky Keyboard Keys  ->  top result is RIGHT
  C2 [Sticky Keyboard Keys] similarity=0.5784: If keyboard keys stick or repeat, turn the laptop off and clean under the keys with compressed air.
  C1 [BitLocker Recovery Key] similarity=0.4111: If your laptop asks for a BitLocker recovery key at startup, call the help desk and read out the key ID shown on the screen.
  C6 [Laptop Freezes] similarity=0.3468: If your laptop freezes, hold the power button for 10 seconds to restart it. If it happens every day, open a ticket.

Question: My new laptop asks for a key
Should come from: BitLocker Recovery Key  ->  top result is WRONG
  C2 [Sticky Keyboard Keys] similarity=0.5799: If keyboard keys stick or repeat, turn the laptop off and clean under the keys with compressed air.
  C1 [BitLocker Recovery Key] similarity=0.5672: If your laptop asks for a BitLocker recovery key at startup, call the help desk and read out the key ID shown on the scr